In [25]:
import pandas as pd 

In [45]:
df = pd.read_csv("IMDB Dataset.csv")

In [46]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [47]:
df.shape

(50000, 2)

In [48]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [49]:
df.drop_duplicates(inplace=True)

In [50]:
df.shape #the shape of our data is changed after removing dublicate values

(49582, 2)

## Pre-processing

### 1. Converting to lowercase 

In [51]:
df["review"] = df["review"].str.lower()

### 2. Removing the URLs

In [52]:
import re

In [53]:
def remove_urls(text):
    text = re.sub( r"http\S+", "", text) #pattern , replacement , string
    return text

df["review"] = df["review"].apply(remove_urls)

### 3. Remove Punctuations

In [54]:
def remove_punctuations(text):
   text = re.sub(r"[^A-Za-z0-9\s]", "", text)
   return text

df["review"] = df["review"].apply(remove_punctuations)


### 4. Removing HTML tags

In [55]:
def remove_html(text):
   text = re.sub(r"<.*?>", "", text)
   return text

df["review"] = df["review"].apply(remove_html)

### 5. Removing the stopwords

In [56]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\pallavi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\pallavi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\pallavi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [57]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [58]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")
    
    return text

df["review"] = df["review"].apply(remove_stopwords)

    

In [59]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


### 6. Stemming

In [60]:
from nltk.stem import PorterStemmer

In [61]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [62]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


### 7. Encoding 

In [63]:
from sklearn.preprocessing import LabelEncoder

In [64]:
le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [65]:
y = df["sentiment"]

In [66]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

### 8. Vectorization

In [67]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [68]:
tf = TfidfVectorizer(max_features=5000)

X  = tf.fit_transform(df["review"])

In [69]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057161 stored elements and shape (49582, 5000)>

## Dataset & Data Loaders

In [80]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [81]:
X_train.shape

(39665, 5000)

In [82]:
X_test.shape

(9917, 5000)

In [83]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [84]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [85]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float(),
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float(),
)



In [86]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

## Build RNN

In [94]:
import torch.nn as nn
import torch.optim as optim

In [98]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0)
        # 1st value = hidden state of all the timestamps
        # 2nd value = final hidden state of last timestamp 

        out = self.fc(out[:, -1, :])
        return out

In [99]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

## Training the RNN 

In [100]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction

        outputs = model(Xb)

        outputs = torch.sigmoid(outputs.squeeze())

        loss = criterion(outputs, yb) # compute loss
        loss.backward()
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.24809549748897552
epoch = 2/10 and loss = 0.27978241443634033
epoch = 3/10 and loss = 0.2517310678958893
epoch = 4/10 and loss = 0.22465640306472778
epoch = 5/10 and loss = 0.2947295308113098
epoch = 6/10 and loss = 0.21907219290733337
epoch = 7/10 and loss = 0.17571142315864563
epoch = 8/10 and loss = 0.12726037204265594
epoch = 9/10 and loss = 0.15493105351924896
epoch = 10/10 and loss = 0.19395075738430023


In [102]:
# evaluate 

model.eval()

with torch.no_grad():
    correct_vals = 0
    total_vals = 0

    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze_(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        total_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/total_vals*100}")

accuracy = 85.67106988000404
